<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EA%B0%9C%EB%A1%A0_10%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np

문제 1 (단답형 주관식)

시그모이드 함수를 깊은 신경망(Deep Learning)의 은닉층에서 잘 사용하지 않는 이유(치명적인 단점)와, ReLU가 이 문제를 어떻게 해결하는지 1~2줄로 간략히 서술하시오.


시그모이드는 층이 깊어질수록 역전파 시 경사가 0에 수렴이 되어 학습이 멈추는 경사 소실 문제가 발생하기 때문이다.

하지만 ReLU는 입력값이 양수이면 미분값이 항상 1로 동일하기에 경사가 소실되지 않고 깊은 층까지 잘 전달되어 안정적인 학습이 가능해진다.

문제 2 (단답형 주관식)

60,000개의 대용량 MNIST 데이터를 학습할 때, 모든 데이터를 한꺼번에 학습시키지 않고 100개씩 작은 묶음(Mini-Batch)으로 나누어 학습시켰습니다.

- 2-1) 이렇게 미니 배치(Mini-Batch)로 나누어 학습하는 주된 이점은 무엇인가요?
- 2-2) 파이토치의 DataLoader에서 shuffle=True 옵션을 사용하는 이유는 무엇인가요?

2-1) 전체 데이터를 한 번에 학습시키는 것보다 메모리가 훨씬 효율적이고 학습 과정에서 적절한 노이즈를 주어 모델이 국소 최적점에 빠지는 것을 방지하고 일반화 성능을 높여준다.

2-2) 모델이 데이터 순서 자체를 외우는 것을 방지하여 일반화 성능을 향상시키기 위함이다.

문제 3 (실습 문제 - 코드 빈칸 채우기)

digits 데이터셋 분류를 위한 3계층 신경망(AdvancedClassifier)을 정의하는 코드입니다.

배치 정규화(BatchNorm)와 드롭아웃(Dropout)을 포함하여, 4개의 빈칸 (# TODO: ...)을 채워 모델을 완성하시오.


In [ ]:
import torch.nn as nn

class AdvancedClassifier(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, num_classes, dropout_rate=0.4):
        super(AdvancedClassifier, self).__init__()

        # 첫 번째 블록: Linear -> BatchNorm -> ReLU -> Dropout
        self.fc1 = nn.Linear(input_size, hidden1_size)

        # TODO: 1. 첫 번째 은닉층(hidden1_size)을 위한 배치 정규화 계층(self.bn1) 정의
        self.bn1 = nn.BatchNorm1d(hidden1_size)

        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)

        # 두 번째 블록
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)

        # TODO: 2. 두 번째 은닉층(hidden2_size)을 위한 배치 정규화 계층(self.bn2) 정의
        self.bn2 = nn.BatchNorm1d(hidden2_size)

        self.relu2 = nn.ReLU()

        # TODO: 3. 두 번째 은닉층을 위한 드롭아웃 계층(self.dropout2) 정의
        self.dropout2 = nn.Dropout(dropout_rate)

        # 출력층
        self.fc3 = nn.Linear(hidden2_size, num_classes)

    def forward(self, x):
        # 첫 번째 블록
        x = self.fc1(x)
        x = self.bn1(x) # TODO: 4. 첫 번째 배치 정규화 적용
        x = self.relu1(x)
        x = self.dropout1(x)

        # 두 번째 블록 (순서대로 적용)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        # 출력층 (logits)
        logits = self.fc3(x)
        return logits

# --- 테스트 코드 (수정 불필요) ---
input_size = 64
num_classes = 10
model = AdvancedClassifier(input_size, 128, 64, num_classes)
print(model)

# (배치 크기 2, 특성 64)의 더미 입력으로 테스트
# (BatchNorm은 배치 단위로 동작하므로 배치 크기가 1보다 커야 함)
dummy_input = torch.randn(2, input_size)
output = model(dummy_input)
print(f"\n출력 크기: {output.shape}")

AdvancedClassifier(
  (fc1): Linear(in_features=64, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.4, inplace=False)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

출력 크기: torch.Size([2, 10])


문제 4 (실습 문제 - 코드 빈칸 채우기)

torchvision을 사용하여 MNIST 훈련 데이터셋을 로드하고, DataLoader를 생성하는 코드입니다.

[요구사항]

데이터를 텐서(Tensor)로 변환합니다.

데이터를 view(-1)를 이용해 1차원 벡터(784,)로 펼칩니다.
batch_size와 shuffle 옵션을 설정하여 train_loader를 생성합니다.

4개의 빈칸 (# TODO: ...)을 채우시오.

In [ ]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# TODO: 1. ToTensor()와 Lambda를 사용하여 28x28 이미지를 784차원 벡터로 변환하는 'transform' 정의
# (힌트: transforms.Compose 사용)
transform = transforms.Compose([
    transforms.ToTensor(),             # 1. 텐서로 변환 (0~1 정규화 포함)
    transforms.Lambda(lambda x: x.view(-1))  # 2. 1차원 벡터로 펼치기
])

# --- MNIST 데이터셋 로드 (수정 불필요) ---
train_set = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
# -----------------------------------

# TODO: 3. 배치 크기(batch_size)를 100으로 설정
batch_size = 100

# TODO: 4. 훈련용 'train_loader' 생성 (batch_size 적용 및 데이터 셔플 활성화)
train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)


# --- 결과 확인 (수정 불필요) ---
print(f"데이터셋 개수: {len(train_set)}")
print(f"데이터로더 배치 개수: {len(train_loader)}")

# 1개 배치(Batch) 샘플링
images, labels = next(iter(train_loader))
print(f"\n1개 배치의 이미지 Shape: {images.shape}") # [100, 784]
print(f"1개 배치의 레이블 Shape: {labels.shape}")   # [100]

데이터셋 개수: 60000
데이터로더 배치 개수: 600

1개 배치의 이미지 Shape: torch.Size([100, 784])
1개 배치의 레이블 Shape: torch.Size([100])


문제 5 (실습 문제 - 코드 빈칸 채우기)

sklearn.metrics를 사용하여 테스트 데이터(y_true)와 모델 예측값(y_pred) 사이의 성능을 평가하는 코드입니다.

3개의 빈칸 (# TODO: ...)을 채워 전체 정확도(Accuracy)와 'macro' 평균 정밀도(Precision)를 계산하시오.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score

# --- 가상의 테스트 결과 (수정 불필요) ---
# y_true: 실제 정답
y_true = np.array([0, 1, 2, 2, 1, 0, 9, 8])
# y_pred: 모델의 예측
y_pred = np.array([0, 1, 2, 1, 1, 0, 9, 9])
# (2는 1로, 8은 9로 잘못 예측)
# ------------------------------------

# TODO: 1. 'accuracy_score'를 사용하여 전체 정확도 'accuracy' 계산
accuracy = accuracy_score(y_true, y_pred)

# TODO: 2. 'precision_score'를 사용하여 정밀도 'precision_macro' 계산
# (힌트: 모든 클래스를 동일하게 취급하기 위해 average='macro' 옵션 사용)
precision_macro = precision_score(y_true, y_pred, average='macro')


# --- 결과 확인 (수정 불필요) ---
# (정확도: 6/8 = 0.75)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (Macro Avg): {precision_macro:.4f}")

# TODO: 3. classification_report() 함수를 사용하여 상세 보고서 출력
report = classification_report(y_true, y_pred, zero_division=0)
print("\n[Classification Report]\n", report)

Accuracy: 0.7500
Precision (Macro Avg): 0.6333

[Classification Report]
 None


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


문제 6 (실습 문제 - 코드 빈칸 채우기)

학습이 완료된 모델의 가중치(state_dict)를 저장하고,
나중에 이 가중치를 새로운 모델 인스턴스에 불러오는 코드입니다.

4개의 빈칸 (# TODO: ...)을 채우시오.

In [ ]:
import torch
import torch.nn as nn

# --- 모델 정의 및 학습된 가상 모델 (수정 불필요) ---
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(10, 2)
model_trained = Net()
# (실제로는 학습을 통해 파라미터가 업데이트 되었다고 가정)
# ----------------------------------------------------

MODEL_PATH = 'my_model.pth' # 저장 경로

# --- 모델 저장 ---
# TODO: 1. 'model_trained'의 'state_dict'를 'MODEL_PATH'에 저장
torch.save(model_trained.state_dict(), MODEL_PATH)


# --- 모델 불러오기 ---

# TODO: 2. 'Net' 클래스의 새로운 인스턴스 'model_new' 생성
model_new = Net()

# TODO: 3. 'MODEL_PATH'로부터 'state_dict' 불러오기
state_dict = torch.load(MODEL_PATH)

# TODO: 4. 'model_new'에 불러온 'state_dict' 적용
model_new.load_state_dict(state_dict)

# (평가 모드로 설정)
model_new.eval()

# --- 결과 확인 (수정 불필요) ---
print("모델 저장 및 불러오기 완료.")
print("\n새 모델 구조:")
print(model_new)

# 두 모델의 파라미터가 동일한지 확인
match = all(
    torch.equal(p1, p2)
    for p1, p2 in zip(model_trained.parameters(), model_new.parameters())
)
print(f"\n두 모델의 파라미터 일치 여부: {match}")

모델 저장 및 불러오기 완료.

새 모델 구조:
Net(
  (fc): Linear(in_features=10, out_features=2, bias=True)
)

두 모델의 파라미터 일치 여부: True


In [ ]:
# eos